# K2 Horizon 7B FP8 on Amazon SageMaker AI

Deploy K2 Horizon 7B FP8 in your AWS account, send messages, assemble streamed responses, use a synthetic inventory tool and clean up the example resources.

Marketplace publication is in progress. Run this notebook once the product and its regional model package are available to your account.

Choose a deployment configuration supported by the selected Marketplace version:

| Profile | Total context | Instance | GPUs |
| --- | ---: | --- | --- |
| `32k` | 32,768 tokens | `ml.g6.2xlarge` | 1 NVIDIA L4 |

This configuration uses FP8 weights, BF16 activations and KV cache, and one active sequence. The total context includes formatted messages, tool definitions, generated reasoning and the final answer.

Prerequisites: accept the AWS Marketplace product terms, use a SageMaker notebook environment with `boto3` and SageMaker Python SDK v2 installed. Copy the regional model package ARN from the subscribed product’s AWS Marketplace deployment instructions into `PACKAGE_ARN`, or set the `K2_MODEL_PACKAGE_ARN` environment variable. The notebook uses its AWS credentials, Region and execution role automatically. Outside SageMaker, configure the standard SDK credential chain and set `SAGEMAKER_EXECUTION_ROLE_ARN`. Your identity needs SageMaker deployment/invocation permissions and permission to pass that role. Check the selected instance quota and regional capacity before deployment.

There is no model software fee. AWS infrastructure charges apply while the endpoint runs. This notebook creates resources in your account and removes them after the example. See the cleanup instructions below.

Model: [K2 Horizon 7B FP8](https://huggingface.co/IFM/K2-Horizon-7B-FP8). License: [Apache 2.0](https://www.apache.org/licenses/LICENSE-2.0). Support: support@ifm.ai.


In [ ]:
import os
import boto3
import sagemaker

DEPLOYMENT_PROFILE = "32k"  # Supported profiles: 32k
AWS_PROFILE = None  # Uses the normal SDK credential chain
session = boto3.Session(profile_name=AWS_PROFILE)
REGION = session.region_name or "eu-north-1"
session = boto3.Session(profile_name=AWS_PROFILE, region_name=REGION)
sagemaker_session = sagemaker.Session(boto_session=session)
# Copy the model package ARN for this Region from the subscribed product's
# AWS Marketplace deployment instructions, or set K2_MODEL_PACKAGE_ARN.
PACKAGE_ARN = os.environ.get("K2_MODEL_PACKAGE_ARN", "")
if not PACKAGE_ARN:
    raise ValueError("Set PACKAGE_ARN to the regional model package ARN from AWS Marketplace before deploying.")
arn_parts = PACKAGE_ARN.split(":", 5)
if (len(arn_parts) != 6 or arn_parts[0] != "arn" or arn_parts[2] != "sagemaker"
        or arn_parts[3] != REGION or not arn_parts[5].startswith("model-package/")):
    raise ValueError("Use the SageMaker model package ARN for the selected Region: " + REGION)
EXECUTION_ROLE_ARN = os.environ.get("SAGEMAKER_EXECUTION_ROLE_ARN")
if EXECUTION_ROLE_ARN is None:
    EXECUTION_ROLE_ARN = sagemaker.get_execution_role(sagemaker_session=sagemaker_session)
EXISTING_ENDPOINT = None  # Optional endpoint you own; the notebook preserves it


## Client and deployment helpers

The default request reserves 16,384 generation tokens, including reasoning. Keep input plus output within the selected total context limit. An output limit is an allowance, not a target: check the finish reason and handle incomplete answers. The separate reasoning-token usage breakdown is unreliable.

Streaming responses carry SSE records inside AWS EventStream payloads. The helper reassembles records across payload boundaries. Keep reasoning content and tool-call identifiers when replaying a conversation. The inventory tool below uses synthetic data and does not contact an external service.


In [ ]:
"""K2 Horizon 7B FP8 SageMaker package deployment and inference example.

Uses boto3's credential chain. No keys are embedded. A deployment starts billed
compute; run cleanup when finished. No external tool service is executed.
"""
import json
import re
import time
import uuid
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError


def request(messages, **kwargs):
    payload=dict(model='k2-horizon-7b', messages=messages, max_tokens=16384,
                 temperature=1.0, top_p=0.95,
                 chat_template_kwargs={'reasoning_effort':'high','tool_call_format':'xml'})
    payload.update(kwargs)
    return payload


def replay(message):
    history={key:message[key] for key in ['role','content','tool_calls'] if key in message}
    history['reasoning_content']=message.get('reasoning_content',message.get('reasoning','')) or ''
    return history


def invoke(runtime, endpoint, payload):
    response=runtime.invoke_endpoint(EndpointName=endpoint, ContentType='application/json',
                                    Accept='application/json', Body=json.dumps({**payload,'stream':False}).encode())
    body=json.loads(response['Body'].read())
    choice=body['choices'][0]
    if choice['finish_reason']=='length':
        raise RuntimeError('Output budget exhausted; this is not a complete answer. Review prompt/output budgets.')
    return body


def stream(runtime, endpoint, payload):
    response=runtime.invoke_endpoint_with_response_stream(EndpointName=endpoint,
        ContentType='application/json',Accept='text/event-stream',Body=json.dumps({**payload,'stream':True,'stream_options':{'include_usage':True}}).encode())
    parts=[]
    for event in response['Body']:
        if 'PayloadPart' not in event:raise RuntimeError('Streaming error: '+str(event))
        parts.append(event['PayloadPart']['Bytes'])
    # Decode after joining: UTF-8 characters and SSE records may cross AWS chunks.
    text=b''.join(parts).decode('utf-8').replace('\r\n','\n')
    message={'role':'assistant','content':'','reasoning_content':''};calls={};finish=None;done=False;usage=None
    for block in text.split('\n\n'):
        data='\n'.join(line[5:].lstrip() for line in block.splitlines() if line.startswith('data:'))
        if not data:continue
        if done:raise RuntimeError('Unexpected data after stream completion')
        if data=='[DONE]':done=True;continue
        chunk=json.loads(data)
        if 'error' in chunk:raise RuntimeError(str(chunk['error']))
        if chunk.get('usage') is not None:usage=chunk['usage']
        for choice in chunk.get('choices',[]):
            delta=choice.get('delta',{})
            message['content']+=delta.get('content') or ''
            message['reasoning_content']+=delta.get('reasoning_content',delta.get('reasoning')) or ''
            for call in delta.get('tool_calls',[]):
                target=calls.setdefault(call['index'],{'id':'','type':'function','function':{'name':'','arguments':''}})
                target['id']+=call.get('id') or ''
                for key in ['name','arguments']:target['function'][key]+=call.get('function',{}).get(key) or ''
            finish=choice.get('finish_reason') or finish
    if not done or not finish:raise RuntimeError('Incomplete response stream')
    if finish=='length':raise RuntimeError('Stream exhausted its output budget before a complete answer')
    if calls:message['tool_calls']=[calls[i] for i in sorted(calls)]
    return {'message':message,'finish_reason':finish,'usage':usage}


def demonstrate(session, endpoint):
    def check_answer(message, expected):
        assert re.search(r'(?<!\d)'+str(expected)+r'(?!\d)',message.get('content') or ''), message
    runtime=session.client('sagemaker-runtime',config=Config(read_timeout=300,retries={'max_attempts':0}))
    first=[{'role':'user','content':'What is 6 + 1? Answer briefly.'}]
    chat=invoke(runtime,endpoint,request(first))
    assert chat['choices'][0]['finish_reason']=='stop'
    check_answer(chat['choices'][0]['message'],7)
    history=first+[replay(chat['choices'][0]['message']),{'role':'user','content':'Double your previous answer. Answer briefly.'}]
    followup=stream(runtime,endpoint,request(history))
    assert followup['finish_reason']=='stop'
    check_answer(followup['message'],14)
    history+=[replay(followup['message']),{'role':'user','content':'Add 3 to that result. Answer briefly.'}]
    third=invoke(runtime,endpoint,request(history))
    assert third['choices'][0]['finish_reason']=='stop'
    check_answer(third['choices'][0]['message'],17)
    tools=[{'type':'function','function':{'name':'get_inventory','description':'Look up available product quantity.',
        'parameters':{'type':'object','properties':{'sku':{'type':'string'}},'required':['sku'],'additionalProperties':False}}}]
    messages=[{'role':'system','content':'Use the inventory tool for stock quantities. Do not guess inventory levels.'},
              {'role':'user','content':'How many units of SKU DEMO-7B are available?'}]
    tool=stream(runtime,endpoint,request(messages,tools=tools,tool_choice='auto'))
    assert tool['finish_reason']=='tool_calls'
    call,=tool['message']['tool_calls']
    assert call['id'], 'A tool call must have an ID for result replay'
    assert call['function']['name']=='get_inventory' and json.loads(call['function']['arguments'])=={'sku':'DEMO-7B'}
    # Synthetic fixture only. Applications must authorize/validate real tools.
    messages+=[replay(tool['message']),{'role':'tool','tool_call_id':call['id'],
               'content':json.dumps({'sku':'DEMO-7B','available_quantity':17,'source':'synthetic-example'})}]
    answer=invoke(runtime,endpoint,request(messages,tools=tools,tool_choice='auto'))
    assert answer['choices'][0]['finish_reason']=='stop'
    check_answer(answer['choices'][0]['message'],17)
    return {'chat':chat,'streamed_followup':followup,'history_replay':third,'streamed_tool_call':tool,'tool_result':answer}


def long_document_demo(session, endpoint):
    runtime=session.client('sagemaker-runtime',config=Config(read_timeout=300,retries={'max_attempts':0}))
    document=('hello '*6000+'\nThe shipment verification code is SAGE-842.\n'+'hello '*6000)
    payload=request([{'role':'user','content':
        'Find the shipment verification code in this synthetic document. Return only the code.\n'+document}],
        max_tokens=16384,seed=7)
    response=invoke(runtime,endpoint,payload)
    choice=response['choices'][0]
    assert choice['finish_reason']=='stop' and 'SAGE-842' in choice['message']['content']
    assert 12000 <= response['usage']['prompt_tokens'] < 15360, response['usage']
    return response


PROFILES = {'32k': {'instance_type': 'ml.g6.2xlarge', 'context_tokens': 32768, 'tensor_parallel': 1}}


def deploy(session, package_arn, role_arn, profile='32k', name=None, resources=None):
    if profile not in PROFILES:
        raise ValueError('Choose a supported profile: ' + ', '.join(PROFILES))
    recipe=PROFILES[profile]
    sm=session.client('sagemaker');package=sm.describe_model_package(ModelPackageName=package_arn)
    if package['ModelPackageStatus']!='Completed':raise RuntimeError('Package is not ready')
    if recipe['instance_type'] not in package['InferenceSpecification']['SupportedRealtimeInferenceInstanceTypes']:
        raise RuntimeError('Package does not support the selected instance type')
    name=name or 'k2-horizon-7b-fp8-example-'+uuid.uuid4().hex[:8]
    resources={} if resources is None else resources
    if resources:raise ValueError('Pass an empty resource dictionary for a new deployment')
    resources.update(model=name,config=name+'-'+profile,endpoint=name+'-test')
    print('Keep these names for cleanup:',json.dumps(resources),flush=True)
    try:
        sm.create_model(ModelName=name,ExecutionRoleArn=role_arn,EnableNetworkIsolation=True,
            PrimaryContainer={'ModelPackageName':package_arn,'Environment':{'K2_SERVED_MODEL_NAME':'k2-horizon-7b',
                'K2_TENSOR_PARALLEL_SIZE':str(recipe['tensor_parallel']),
                'K2_MAX_MODEL_LEN':str(recipe['context_tokens']),'K2_MAX_NUM_SEQS':'1',
                'K2_MAX_NUM_BATCHED_TOKENS':'512','K2_ENFORCE_EAGER':'1',
                'K2_GPU_MEMORY_UTILIZATION':'0.90','VLLM_ENABLE_CUDA_COMPATIBILITY':'0',
                'HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1','HF_DATASETS_OFFLINE':'1'}})
        resources['model_created']=True
        sm.create_endpoint_config(EndpointConfigName=resources['config'],ProductionVariants=[{
            'VariantName':'AllTraffic','ModelName':name,'InstanceType':recipe['instance_type'],'InitialInstanceCount':1,
            'InitialVariantWeight':1.0,'InferenceAmiVersion':'al2023-ami-sagemaker-inference-gpu-4-1',
            'ModelDataDownloadTimeoutInSeconds':1800,'ContainerStartupHealthCheckTimeoutInSeconds':900}])
        resources['config_created']=True
        # Newly created configurations can take a short time to become visible.
        for attempt in range(12):
            try:
                sm.create_endpoint(EndpointName=resources['endpoint'],EndpointConfigName=resources['config'])
                break
            except ClientError as error:
                message=error.response['Error'].get('Message','').lower()
                if (attempt == 11 or error.response['Error']['Code'] != 'ValidationException'
                        or 'endpoint configuration' not in message or 'could not find' not in message):
                    raise
                time.sleep(5)
        resources['endpoint_created']=True
        for _ in range(80):
            state=sm.describe_endpoint(EndpointName=resources['endpoint'])
            print('Endpoint:',state['EndpointStatus'],flush=True)
            if state['EndpointStatus']=='InService':return resources
            if state['EndpointStatus']=='Failed':raise RuntimeError(state.get('FailureReason','Deployment failed'))
            time.sleep(30)
        raise TimeoutError('Deployment did not finish within 40 minutes')
    except BaseException:
        cleanup(session,resources)
        raise


def absent_resource(error):
    detail=error.response['Error']
    return detail['Code']=='ValidationException' and any(
        phrase in detail['Message'].lower() for phrase in ('could not find','does not exist','not found'))


def cleanup(session, resources):
    sm=session.client('sagemaker')
    if resources.get('endpoint_created'):
        # SageMaker rejects deletion while endpoint creation is in progress.
        # Keep the resource markers intact if this bounded wait expires.
        for _ in range(60):
            try:state=sm.describe_endpoint(EndpointName=resources['endpoint'])
            except ClientError as error:
                if absent_resource(error):
                    state=None
                    break
                raise
            if state['EndpointStatus']!='Creating':break
            print('Cleanup waiting for SageMaker creation lock:',resources['endpoint'],flush=True)
            time.sleep(15)
        else:raise RuntimeError('SageMaker still holds the creation lock; retry cleanup and inspect AWS before leaving')
        if state is not None and state['EndpointStatus']!='Deleting':
            sm.delete_endpoint(EndpointName=resources['endpoint'])
        for _ in range(40):
            try:sm.describe_endpoint(EndpointName=resources['endpoint'])
            except ClientError as error:
                if absent_resource(error):break
                raise
            time.sleep(15)
        else:raise RuntimeError('Endpoint deletion not confirmed; inspect the AWS console before leaving')
        resources['endpoint_created']=False
    if resources.get('config_created'):
        try:sm.delete_endpoint_config(EndpointConfigName=resources['config'])
        except ClientError as error:
            if not absent_resource(error):raise
        resources['config_created']=False
    if resources.get('model_created'):
        try:sm.delete_model(ModelName=resources['model'])
        except ClientError as error:
            if not absent_resource(error):raise
        resources['model_created']=False
    print('Example resources cleaned up.',flush=True)


def batch_demo(session, package_arn, role_arn, bucket, resources, name=None):
    """Run one generated JSON request on a temporary 32K batch model."""
    if resources:raise ValueError('Pass an empty resource dictionary for a new batch job')
    sm=session.client('sagemaker');s3=session.client('s3')
    name=name or 'k2-horizon-7b-fp8-batch-'+uuid.uuid4().hex[:8]
    prefix='k2-horizon-examples/'+name
    resources.update(model=name,job=name,bucket=bucket,input_key=prefix+'/input/request.json',
                     output_key=prefix+'/output/request.json.out')
    print('Keep these names for cleanup:',json.dumps(resources),flush=True)
    try:
        sample=request([{'role':'user','content':'Calculate 23 times 47. Return only the integer result.'}],stream=False)
        s3.put_object(Bucket=bucket,Key=resources['input_key'],Body=json.dumps(sample).encode(),ContentType='application/json')
        resources['input_created']=True
        sm.create_model(ModelName=name,ExecutionRoleArn=role_arn,EnableNetworkIsolation=True,
            PrimaryContainer={'ModelPackageName':package_arn,'Environment':{
                'K2_SERVED_MODEL_NAME':'k2-horizon-7b','K2_MAX_MODEL_LEN':'32768','K2_TENSOR_PARALLEL_SIZE':'1',
                'K2_MAX_NUM_SEQS':'1','K2_MAX_NUM_BATCHED_TOKENS':'512','K2_ENFORCE_EAGER':'1',
                'K2_GPU_MEMORY_UTILIZATION':'0.90','HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1','HF_DATASETS_OFFLINE':'1',
                'VLLM_ENABLE_CUDA_COMPATIBILITY':'1','VLLM_CUDA_COMPATIBILITY_PATH':'/usr/local/cuda-13.0/compat',
                'LD_LIBRARY_PATH':'/usr/local/cuda-13.0/compat:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:/usr/local/nvidia/lib'}})
        resources['model_created']=True
        sm.create_transform_job(TransformJobName=name,ModelName=name,MaxConcurrentTransforms=1,MaxPayloadInMB=1,
            ModelClientConfig={'InvocationsTimeoutInSeconds':300,'InvocationsMaxRetries':0},
            TransformInput={'DataSource':{'S3DataSource':{'S3DataType':'S3Prefix','S3Uri':'s3://'+bucket+'/'+resources['input_key']}},
                            'ContentType':'application/json','CompressionType':'None','SplitType':'None'},
            TransformOutput={'S3OutputPath':'s3://'+bucket+'/'+prefix+'/output/', 'Accept':'application/json','AssembleWith':'None'},
            TransformResources={'InstanceType':'ml.g6.2xlarge','InstanceCount':1,'TransformAmiVersion':'al2-ami-sagemaker-batch-gpu-535'})
        resources['job_created']=True
        for _ in range(80):
            state=sm.describe_transform_job(TransformJobName=name)
            print('Batch job:',state['TransformJobStatus'],flush=True)
            if state['TransformJobStatus']=='Completed':
                response=json.loads(s3.get_object(Bucket=bucket,Key=resources['output_key'])['Body'].read())
                choice=response['choices'][0]
                if choice['finish_reason']!='stop':raise RuntimeError('Batch response is incomplete: '+str(choice['finish_reason']))
                if choice['message']['content'].strip()!='1081':raise RuntimeError('Unexpected answer for the arithmetic example')
                return {'input':sample,'output':response}
            if state['TransformJobStatus'] in ('Failed','Stopped'):
                raise RuntimeError(state.get('FailureReason','Batch job stopped'))
            time.sleep(30)
        raise TimeoutError('Batch job did not finish within 40 minutes')
    finally:
        cleanup_batch(session,resources)


def cleanup_batch(session, resources):
    """Stop active work and remove only this example's model and exact S3 keys."""
    sm=session.client('sagemaker')
    if resources.get('job_created'):
        state=sm.describe_transform_job(TransformJobName=resources['job'])
        if state['TransformJobStatus']=='InProgress':
            sm.stop_transform_job(TransformJobName=resources['job'])
        for _ in range(60):
            state=sm.describe_transform_job(TransformJobName=resources['job'])
            if state['TransformJobStatus'] in ('Completed','Failed','Stopped'):break
            time.sleep(15)
        else:raise RuntimeError('Batch termination not confirmed; retain resources and retry cleanup_batch')
        resources['job_created']=False
    if resources.get('model_created'):
        try:sm.delete_model(ModelName=resources['model'])
        except ClientError as error:
            if not absent_resource(error):raise
        resources['model_created']=False
    if resources.get('input_created'):
        s3=session.client('s3')
        for key in ('input_key','output_key'):
            s3.delete_object(Bucket=resources['bucket'],Key=resources[key])
        resources['input_created']=False
    print('Batch compute stopped and example resources cleaned up.',flush=True)


## Run and clean up

The following cell deploys the selected profile, runs the examples and deletes the resources it creates in a `finally` block. Instance allocation may fail when regional capacity is unavailable. The examples demonstrate usage; they do not establish task quality or production throughput.

Keep the notebook running until cleanup confirms deletion. If the kernel or connection is interrupted, inspect SageMaker endpoints in the selected region and delete the example endpoint, then its endpoint configuration and model. A local timeout does not stop AWS billing. Cleanup may have to wait for SageMaker to release its creation lock.

When `EXISTING_ENDPOINT` is set, the cell only invokes that endpoint and leaves its lifecycle to you.


In [ ]:
resources = None
try:
    if EXISTING_ENDPOINT is None:
        if EXECUTION_ROLE_ARN.startswith("REPLACE_"):
            raise ValueError("Set your execution role ARN before deploying")
        resources = {}
        deploy(session, PACKAGE_ARN, EXECUTION_ROLE_ARN, profile=DEPLOYMENT_PROFILE, resources=resources)
        endpoint = resources["endpoint"]
    else:
        endpoint = EXISTING_ENDPOINT
    results = demonstrate(session, endpoint)
    long_document = long_document_demo(session, endpoint)
    print("Long-document result:", long_document["choices"][0]["message"]["content"], long_document["usage"])
    print("Chat:", results["chat"]["choices"][0]["message"]["content"])
    print("Streamed follow-up:", results["streamed_followup"]["message"]["content"])
    print("History replay:", results["history_replay"]["choices"][0]["message"]["content"])
    print("Tool result:", results["tool_result"]["choices"][0]["message"]["content"])
finally:
    if resources is not None:
        cleanup(session, resources)

## Batch transform example

Batch transform processes a generated JSON input from S3 and writes a JSON response to S3. This example uses the 32K profile on one `ml.g6.2xlarge` instance. It does not create a persistent endpoint. A transform quota and available capacity are required separately from endpoint quotas.

The SDK selects or creates your account’s standard SageMaker bucket in the selected Region; you can override `BATCH_BUCKET` with another bucket you own. Your identity and SageMaker execution role need access to the example prefix. This cell generates the input; no external dataset is needed. The next cell runs one batch job by default; set `RUN_BATCH=False` to skip it. AWS infrastructure and S3 charges apply. The cleanup function stops an unfinished job, waits for compute to terminate, deletes the model and removes the two exact example objects. A versioned bucket can retain object versions under its lifecycle rules; the bucket is preserved.

Keep `batch_resources` if a cleanup attempt fails, then retry `cleanup_batch(session, batch_resources)` and verify the job status in SageMaker. Completed job metadata remains available in AWS.


In [ ]:
RUN_BATCH = True
BATCH_BUCKET = None  # Uses the account's regional SageMaker bucket
batch_resources = {}
if RUN_BATCH:
    BATCH_BUCKET = BATCH_BUCKET or sagemaker_session.default_bucket()
    try:
        batch_result = batch_demo(session, PACKAGE_ARN, EXECUTION_ROLE_ARN, BATCH_BUCKET, batch_resources)
        print("Batch answer:", batch_result["output"]["choices"][0]["message"]["content"])
    finally:
        cleanup_batch(session, batch_resources)
